# AGH DS Laboratory 6 - Actor Model with Ray Framework

## Introduction

Ray is a general-purpose framework for programming a cluster made by UC Berkeley's RISELab. It enables developers to easily parallelize their Python applications or build new ones, and run them at any scale, from a laptop to a large cluster. It also provides a highly flexible, yet minimalist and easy to use API. 

#### Documentation Reference Links:

Ray documentation: https://docs.ray.io/en/latest/ \
Ray GitHub: https://github.com/ray-project/ray \
Ray Core: https://docs.ray.io/en/latest/ray-core/walkthrough.html \
Ray Client: https://docs.ray.io/en/latest/cluster/running-applications/job-submission/ray-client.html \

***
## Part 1 - Remote Functions

This script is too slow, and the computation is embarrassingly parallel. In this exercise, you will use Ray to execute the functions in parallel to speed it up.

The standard way to turn a Python function into a remote function is to add the `@ray.remote` decorator. Here is an example.

```python
# A regular Python function.
def regular_function(x):
    return x + 1

# A Ray remote function.
@ray.remote
def remote_function(x):
    return x + 1
```

The differences are the following:

1. **Invocation:** The regular version is called with `regular_function(1)`, whereas the remote version is called with `remote_function.remote(1)`.
2. **Return values:** `regular_function` immediately executes and returns `1`, whereas `remote_function` immediately returns an object ID (a future) and then creates a task that will be executed on a worker process. The result can be obtained with `ray.get`.
    ```python
    >>> regular_function(0)
    1
    
    >>> remote_function.remote(0)
    ObjectID(1c80d6937802cd7786ad25e50caf2f023c95e350)
    
    >>> ray.get(remote_function.remote(0))
    1
    ```
3. **Parallelism:** Invocations of `regular_function` happen **serially**, for example
    ```python
    # These happen serially.
    for _ in range(4):
        regular_function(0)
    ```
    whereas invocations of `remote_function` happen in **parallel**, for example
    ```python
    # These happen in parallel.
    for _ in range(4):
        remote_function.remote(0)
    ```

In [100]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import ray
import time
import numpy as np
from numpy import random
import os
import pickle

import sys

print("Python:", sys.version)
print("Ray:", ray.__version__)

Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:56) [GCC 14.3.0]
Ray: 2.55.1


Start Ray. By default, Ray does not schedule more tasks concurrently than there are CPUs. This example requires four tasks to run concurrently, so we tell Ray that there are four CPUs. Usually this is not done and Ray computes the number of CPUs using `psutil.cpu_count()`. The argument `ignore_reinit_error=True` just ignores errors if the cell is run multiple times.

The call to `ray.init` starts a number of processes.

In [101]:
STUDENT_ID = "kaczmarski_jan"  # WPISZ SWOJE IMIE I NAZWISKO
RAY_ADDRESS = "ray://abb3b46c076a04eeda952e11a94d7d44-2121980027.us-east-1.elb.amazonaws.com:10001"

if ray.is_initialized():
    ray.shutdown()

ray.init(
    #address=RAY_ADDRESS,
    ignore_reinit_error=True,
    namespace=f"lab-ray-{STUDENT_ID}",
)

print("Connected")
print("Student:", STUDENT_ID)
print("Namespace:", ray.get_runtime_context().namespace)
print("Job ID:", ray.get_runtime_context().get_job_id())
print(ray.cluster_resources())


2026-06-02 22:57:12,450	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Connected
Student: kaczmarski_jan
Namespace: lab-ray-kaczmarski_jan
Job ID: 01000000
{'CPU': 12.0, 'node:__internal_head__': 1.0, 'memory': 3580530688.0, 'object_store_memory': 1534513152.0, 'node:172.23.0.2': 1.0}


Create a helloWorld Actor and verify it shows on the dashboard.                      

In [102]:
ACTOR_NAME = f"actor-{STUDENT_ID}"

@ray.remote
class Hello(object):
    def __init__(self, x):
        self.x = x
    
    def set(self, x):
        self.x = x
    
    def get(self):
        return self.x

hello = Hello.options(name=ACTOR_NAME).remote(10)


In [103]:
# This function is a proxy for a more interesting and computationally
# intensive function.

@ray.remote
def slow_function(i):
    print(f"Running in pid {os.getpid()}")
    time.sleep(1)
    return i

**EXERCISE:** The loop below takes too long. The four function calls could be executed in parallel. Instead of four seconds, it should only take one second. Once `slow_function` has been made a remote function, execute these four tasks in parallel by calling `slow_function.remote()`. Then obtain the results by calling `ray.get` on a list of the resulting object IDs.

In [104]:
# Sleep a little to improve the accuracy of the timing measurements below.
# We do this because workers may still be starting up in the background.
time.sleep(2.0)
start_time = time.time()

results = [slow_function.remote(i) for i in range(4)]
for i in range(len(results)):
    results[i] = ray.get(results[i])

end_time = time.time()
duration = end_time - start_time

print('The results are {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(results, duration))

(slow_function pid=14326) Running in pid 14326
The results are [0, 1, 2, 3]. This took 1.1438062191009521 seconds. Run the next cell to see if the exercise was done correctly.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [105]:
assert results == [0, 1, 2, 3], 'Did you remember to call ray.get?'
assert duration < 3.1, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 1, ('The loop took {} seconds. This is too fast.'
                      .format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 1.1438062191009521 seconds.


***Task: Batch Processing***
Let's implement a simple batch image/data processing simulation. In the cell below, you have a function that processes many independent “log chunks”. Each chunk takes about 1 second.

In [106]:
import time
import random
from collections import Counter
import socket

EVENT_TYPES = ["login", "logout", "purchase", "click", "error"]


def process_log_chunk(chunk_id, chunk_size=10_000):
    """
    Simulate processing one chunk of logs.
    """
    time.sleep(1)  # simulate I/O or expensive parsing

    events = [
        random.choice(EVENT_TYPES)
        for _ in range(chunk_size)
    ]

    counts = Counter(events)

    return {
        "chunk_id": chunk_id,
        "counts": dict(counts),
#        "hostname": socket.gethostname(), #uncomment the line for remote version
    }

Now execute it sequentially

In [107]:
start = time.time()

sequential_results = [
    process_log_chunk(i)
    for i in range(8)
]

sequential_time = time.time() - start

print("Sequential time:", sequential_time)
sequential_results[:2]

Sequential time: 8.07950758934021


[{'chunk_id': 0,
  'counts': {'purchase': 1966,
   'login': 2037,
   'error': 2054,
   'click': 1925,
   'logout': 2018}},
 {'chunk_id': 1,
  'counts': {'error': 1993,
   'logout': 2039,
   'purchase': 1978,
   'click': 1964,
   'login': 2026}}]

Next implement task that will execute the processing logic on distributed cluster (name task as *process_log_chunk_remote*)

In [108]:
import time
import random
from collections import Counter
import socket

EVENT_TYPES = ["login", "logout", "purchase", "click", "error"]

@ray.remote
def process_log_chunk_remote(chunk_id, chunk_size=10_000):
    """
    Simulate processing one chunk of logs.
    """
    time.sleep(1)  # simulate I/O or expensive parsing

    events = [
        random.choice(EVENT_TYPES)
        for _ in range(chunk_size)
    ]

    counts = Counter(events)

    return {
        "chunk_id": chunk_id,
        "counts": dict(counts),
        "hostname": socket.gethostname(), #uncomment the line for remote version
    }

Execute the task in parallel - show the time difference in processing compared to the sequential version.

In [109]:
start = time.time()

results = [
    process_log_chunk_remote.remote(i)
    for i in range(8)
]

for i in range(len(results)):
    results[i] = ray.get(results[i])

paralel_time = time.time() - start

print("time:", paralel_time)
results[:2]

time: 1.1275486946105957


[{'chunk_id': 0,
  'counts': {'click': 1973,
   'error': 1979,
   'login': 2068,
   'logout': 1957,
   'purchase': 2023},
  'hostname': 'efcab0f30bdc'},
 {'chunk_id': 1,
  'counts': {'login': 2025,
   'error': 1935,
   'logout': 1960,
   'purchase': 2013,
   'click': 2067},
  'hostname': 'efcab0f30bdc'}]

As example of an anti-pattern, execute tasks in sequential manner using the ray framework, print processing time

In [110]:
start = time.time()

seq_ray_results = []
for i in range(8):
    obj = process_log_chunk_remote.remote(i)
    seq_ray_results.append(ray.get(obj))

seq_ray_time = time.time() - start

print("time:", seq_ray_time)
seq_ray_results[:2]

time: 8.122916460037231


[{'chunk_id': 0,
  'counts': {'logout': 1996,
   'error': 1998,
   'purchase': 1970,
   'login': 1998,
   'click': 2038},
  'hostname': 'efcab0f30bdc'},
 {'chunk_id': 1,
  'counts': {'error': 2044,
   'click': 1926,
   'login': 2024,
   'purchase': 2007,
   'logout': 1999},
  'hostname': 'efcab0f30bdc'}]

Next, compute the total counts of logs on your local Python environment. Keep in mind that this time you do not store anything on the cluster - everything remains on your local notebook environment. It is not the Actor model because you execute everything on the local machine.

In [111]:
from collections import Counter

total_counts = Counter()

for result in results:
    total_counts.update(result["counts"])

total_counts = dict(total_counts)

print(total_counts)
print("Total logs:", sum(total_counts.values()))

{'click': 16094, 'error': 15810, 'login': 15858, 'logout': 16054, 'purchase': 16184}
Total logs: 80000


Finally measure the time for different number of iteration 4,8,16,32 
* When does adding more tasks stop improving runtime?
* How many CPUs does Ray report?                       

In [112]:
print("Total Ray CPUs:", ray.cluster_resources().get("CPU", 0))
print("Free Ray CPUs:", ray.available_resources().get("CPU", 0))

bench_res = []
for iter_cnt in [4, 8, 16, 32]:
    start = time.time()
    
    results = [
        process_log_chunk_remote.remote(i)
        for i in range(iter_cnt)
    ]
    
    for i in range(len(results)):
        results[i] = ray.get(results[i])
    
    paralel_time = time.time() - start
    
    print("iter_cnt:", iter_cnt, "time:", paralel_time)
    bench_res.append(paralel_time)

Total Ray CPUs: 12.0
Free Ray CPUs: 11.0
iter_cnt: 4 time: 1.015016794204712
iter_cnt: 8 time: 1.0170245170593262
iter_cnt: 16 time: 2.032846212387085
iter_cnt: 32 time: 3.05936861038208


***
## Part 2 - Parallel Data Processing with Task Dependencies

**GOAL:** The goal of this exercise is to show how to pass object IDs into remote functions to encode dependencies between tasks.

In this exercise, we construct a sequence of tasks each of which depends on the previous mimicking a data parallel application. Within each sequence, tasks are executed serially, but multiple sequences can be executed in parallel.

In this exercise, you will use Ray to parallelize the computation below and speed it up.

### Concept for this Exercise - Task Dependencies

Suppose we have two remote functions defined as follows.

```python
@ray.remote
def f(x):
    return x
```

Arguments can be passed into remote functions as usual.

```python
>>> x1_id = f.remote(1)
>>> ray.get(x1_id)
1

>>> x2_id = f.remote([1, 2, 3])
>>> ray.get(x2_id)
[1, 2, 3]
```

**Object IDs** can also be passed into remote functions. When the function actually gets executed, **the argument will be a retrieved as a regular Python object**.

```python
>>> y1_id = f.remote(x1_id)
>>> ray.get(y1_id)
1

>>> y2_id = f.remote(x2_id)
>>> ray.get(y2_id)
[1, 2, 3]
```

So when implementing a remote function, the function should expect a regular Python object regardless of whether the caller passes in a regular Python object or an object ID.

**Task dependencies affect scheduling.** In the example above, the task that creates `y1_id` depends on the task that creates `x1_id`. This has the following implications.

- The second task will not be executed until the first task has finished executing.
- If the two tasks are scheduled on different machines, the output of the first task (the value corresponding to `x1_id`) will be copied over the network to the machine where the second task is scheduled.


These are some helper functions that mimic an example pattern of a data parallel application.

**EXERCISE:** You will need to turn all of these functions into remote functions. When you turn these functions into remote function, you do not have to worry about whether the caller passes in an object ID or a regular object. In both cases, the arguments will be regular objects when the function executes. This means that even if you pass in an object ID, you **do not need to call `ray.get`** inside of these remote functions.

In [113]:
@ray.remote
def load_data(filename):
    time.sleep(0.1)
    return np.ones((1000, 100))

@ray.remote
def normalize_data(data):
    time.sleep(0.1)
    return data - np.mean(data, axis=0)

@ray.remote
def extract_features(normalized_data):
    time.sleep(0.1)
    return np.hstack([normalized_data, normalized_data ** 2])

@ray.remote
def compute_loss(features):
    num_data, dim = features.shape
    time.sleep(0.1)
    return np.sum((np.dot(features, np.ones(dim)) - np.ones(num_data)) ** 2)

assert hasattr(load_data, 'remote'), 'load_data must be a remote function'
assert hasattr(normalize_data, 'remote'), 'normalize_data must be a remote function'
assert hasattr(extract_features, 'remote'), 'extract_features must be a remote function'
assert hasattr(compute_loss, 'remote'), 'compute_loss must be a remote function'

**EXERCISE:** The loop below takes too long. Parallelize the four passes through the loop by turning `load_data`, `normalize_data`, `extract_features`, and `compute_loss` into remote functions and then retrieving the losses with `ray.get`.

**NOTE:** You should only use **ONE** call to `ray.get`. For example, the object ID returned by `load_data` should be passed directly into `normalize_data` without needing to be retrieved by the driver.

In [114]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

losses = []
for filename in ['file1', 'file2', 'file3', 'file4']:
    inner_start = time.time()

    data = load_data.remote(filename)
    normalized_data = normalize_data.remote(data)
    features = extract_features.remote(normalized_data)
    loss = compute_loss.remote(features)
    losses.append(loss)
    
    inner_end = time.time()
    
    if inner_end - inner_start >= 0.4:
        raise Exception('You may be calling ray.get inside of the for loop! '
                        'Doing this will prevent parallelism from being exposed. '
                        'Make sure to only call ray.get once outside of the for loop.')

losses = [ray.get(loss) for loss in losses]
print('The losses are {}.'.format(losses) + '\n')
loss = sum(losses)

end_time = time.time()
duration = end_time - start_time

print('The loss is {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(loss, duration))

The losses are [1000.0, 1000.0, 1000.0, 1000.0].

The loss is 4000.0. This took 0.45441102981567383 seconds. Run the next cell to see if the exercise was done correctly.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [115]:
assert loss == 4000
assert duration < 2.8, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 0.4, ('The loop took {} seconds. This is too fast.'
                        .format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.45441102981567383 seconds.


**Note:** If your tests fail and values are closed to reference it is OK (the cluster is slow, and there are communication costs)

***
## Part 3 - Introducing Actors

**Goal:** The goal of this exercise is to show how to create an actor and how to call actor methods.

See the documentation on actors at https://docs.ray.io/en/latest/ray-core/actors.html.

Sometimes you need a "worker" process to have "state". For example, that state might be a neural network, a simulator environment, a counter, or something else entirely. However, remote functions are side-effect free. That is, they operate on inputs and produce outputs, but they don't change the state of the worker they execute on.

Actors are different. When we instantiate an actor, a brand new worker is created, and all methods that are called on that actor are executed on the newly created worker.

This means that with a single actor, no parallelism can be achieved because calls to the actor's methods will be executed one at a time. However, multiple actors can be created and methods can be executed on them in parallel.

### Concepts for this Exercise - Actors

To create an actor, decorate Python class with the `@ray.remote` decorator.

```python
@ray.remote
class Example(object):
    def __init__(self, x):
        self.x = x
    
    def set(self, x):
        self.x = x
    
    def get(self):
        return self.x
```

Like regular Python classes, **actors encapsulate state that is shared across actor method invocations**.

Actor classes differ from regular Python classes in the following ways.
1. **Instantiation:** A regular class would be instantiated via `e = Example(1)`. Actors are instantiated via
    ```python
    e = Example.remote(1)
    ```
    When an actor is instantiated, a **new worker process** is created by a local scheduler somewhere in the cluster.
2. **Method Invocation:** Methods of a regular class would be invoked via `e.set(2)` or `e.get()`. Actor methods are invoked differently.
    ```python
    >>> e.set.remote(2)
    ObjectID(d966aa9b6486331dc2257522734a69ff603e5a1c)
    
    >>> e.get.remote()
    ObjectID(7c432c085864ed4c7c18cf112377a608676afbc3)
    ```
3. **Return Values:** Actor methods are non-blocking. They immediately return an object ID and **they create a task which is scheduled on the actor worker**. The result can be retrieved with `ray.get`.
    ```python
    >>> ray.get(e.set.remote(2))
    None
    
    >>> ray.get(e.get.remote())
    2
    ```

**EXERCISE:** Change the `Foo` class to be an actor class by using the `@ray.remote` decorator.

In [116]:
@ray.remote
class Foo(object):
    def __init__(self):
        self.counter = 0

    def reset(self):
        self.counter = 0

    def increment(self):
        time.sleep(0.5)
        self.counter += 1
        return self.counter

assert hasattr(Foo, 'remote'), 'You need to turn "Foo" into an actor with @ray.remote.'

**EXERCISE:** Change the intantiations below to create two actors by calling `Foo.remote()`.

In [117]:
# Create two Foo objects.
f1 = Foo.remote()
f2 = Foo.remote()

**EXERCISE:** Parallelize the code below. The two actors can execute methods in parallel (though each actor can only execute one method at a time).

In [118]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

# Reset the actor state so that we can run this cell multiple times without
# changing the results.
f1.reset.remote()
f2.reset.remote()

# We want to parallelize this code. However, it is not straightforward to
# make "increment" a remote function, because state is shared (the value of
# "self.counter") between subsequent calls to "increment". In this case, it
# makes sense to use actors.
results = []
for _ in range(5):
    results.append(f1.increment.remote())
    results.append(f2.increment.remote())

results = [ray.get(res) for res in results]

end_time = time.time()
duration = end_time - start_time

print('Success! The example took {} seconds.'.format(duration))

assert not any([isinstance(result, ray.ObjectRef) for result in results]), 'Looks like "results" is {}. You may have forgotten to call ray.get.'.format(results)

Success! The example took 2.6316659450531006 seconds.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [119]:
assert results == [1, 1, 2, 2, 3, 3, 4, 4, 5, 5]

assert duration < 3.2, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))
assert duration > 2.5, ('The experiments ran in {} seconds. This is too '
                        'fast.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 2.6316659450531006 seconds.


**LogAggregator Task** Next, create an actor that will aggregate log events from the *Task Exercise* part of the class. The actor should implement methods such as: 
1. add_chunk_result - adding chunk
2. get_count - total counts
3. get_processed_chunks - current processed chunks
4. get_summary - return num_chunks, num_events and total counts
5. reset - clears all store data


In [120]:
@ray.remote
class LogAggregator(object):
    def __init__(self):
        self.processed = []

    def add_chunk_result(self, chunk_obj):
        self.processed.append(chunk_obj)

    def get_processed_chunks(self):
        return self.processed

    def get_count(self):
        total = Counter()

        for chunk in self.processed:
            total.update(chunk["counts"])

        return dict(total)

    def get_summary(self):
        total_counts = Counter()

        for chunk in self.processed:
            total_counts.update(chunk["counts"])

        return {
            "num_chunks": len(self.processed),
            "num_events": sum(total_counts.values()),
            "total_counts": dict(total_counts),
        }

    def reset(self):
        self.processed = []

It will be named actor it means its name will be unique

*Note* If you want to replace implemantaion of the named actor you need to kill it first (*ray.kill(old_actor, no_restart=True)*)

In [121]:
ACTOR_NAME = f"log-aggregator-{STUDENT_ID}"

try:
    aggregator = ray.get_actor(ACTOR_NAME)
    print(f"Using existing actor: {ACTOR_NAME}")
except ValueError:
    aggregator = LogAggregator.options(name=ACTOR_NAME).remote()
    print(f"Created new actor: {ACTOR_NAME}")

# Optional: reset state before exercise
ray.get(aggregator.reset.remote())

Created new actor: log-aggregator-kaczmarski_jan


Next collect all data, and display summary

In [122]:
import pandas as pd
from pprint import pprint

NUM_CHUNKS = 8
CHUNK_SIZE = 10_000

chunk_refs = [
    process_log_chunk_remote.remote(chunk_id, CHUNK_SIZE)
    for chunk_id in range(NUM_CHUNKS)
]

add_refs = [
    aggregator.add_chunk_result.remote(chunk_ref)
    for chunk_ref in chunk_refs
]

ray.get(add_refs)

summary = ray.get(aggregator.get_summary.remote())

print("Summary:")
pprint(summary)


Summary:
{'num_chunks': 8,
 'num_events': 80000,
 'total_counts': {'click': 16121,
                  'error': 16061,
                  'login': 15925,
                  'logout': 16025,
                  'purchase': 15868}}


Next run more tasks and prove that the state is updated

In [123]:
MORE_CHUNKS = 4
CHUNK_SIZE = 10_000

before = ray.get(aggregator.get_summary.remote())
processed = ray.get(aggregator.get_processed_chunks.remote())
start_id = max((c["chunk_id"] for c in processed), default=-1) + 1

refs = [process_log_chunk_remote.remote(start_id + i, CHUNK_SIZE) for i in range(MORE_CHUNKS)]
ray.get([aggregator.add_chunk_result.remote(ref) for ref in refs])

after = ray.get(aggregator.get_summary.remote())

print("Before:", before)
print("After:", after)

Before: {'num_chunks': 8, 'num_events': 80000, 'total_counts': {'login': 15925, 'purchase': 15868, 'click': 16121, 'logout': 16025, 'error': 16061}}
After: {'num_chunks': 12, 'num_events': 120000, 'total_counts': {'login': 24028, 'purchase': 23815, 'click': 24173, 'logout': 24086, 'error': 23898}}


## Part 4 - More advanced Actors, queuing operations

**GOAL:** The goal of this exercise is to illustrate how to actors queues of different operations

### Concepts for this Exercise - learn different ways for serializing calls on actor 


As the base for the exercise, create the Actor that will implement bank account functionality. 
1. It should keep information regarding the owner, balance, and all operations.
2. The account should support deposit and withdrawal operations (both should have a 1-second sleep time for processing). On successful operations return set with owner, name of operation, amount, balance and hostname (socket.gethostname())
3. Add getters for  balance and history


In [124]:
from typing import List
from enum import Enum
import time

class OperationType(Enum):
    Withdraw = 1
    Deposit = 2

@ray.remote
class BankAccount(object):
    def __init__(self, ownerId: str, initial_balance:float = 0.0):
        self.balance: float = initial_balance
        self.operations: List[Operation] = []
        self.ownerId: str  = ownerId
        
    def deposit(self, amount: int):
        time.sleep(1)
        
        self.balance += amount
        self.operations.append((OperationType.Deposit, amount))
        
        return (self.ownerId, OperationType.Deposit, amount, self.balance, socket.gethostname())

    def withdraw(self, amount: int):
        time.sleep(1)
        if self.balance < amount:
            raise ValueError("amount for withdrawal bigger than account balance")

        self.balance -= amount
        self.operations.append((OperationType.Withdraw, amount))

        return (self.ownerId, OperationType.Withdraw, amount, self.balance, socket.gethostname())

    def get_history(self):
        return self.operations

    def get_balance(self):
        return self.balance
        

The following operations should pass

In [125]:
account = BankAccount.remote("kowalski", initial_balance=100)

start = time.time()

refs = [
    account.deposit.remote(10),
    account.deposit.remote(20),
    account.withdraw.remote(50),
    account.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
print("Results:")
for r in results:
    print(r)

print("Final balance:", ray.get(account.get_balance.remote()))
print("History:", ray.get(account.get_history.remote()))

Duration: 4.367348670959473
Results:
('kowalski', <OperationType.Deposit: 2>, 10, 110, 'efcab0f30bdc')
('kowalski', <OperationType.Deposit: 2>, 20, 130, 'efcab0f30bdc')
('kowalski', <OperationType.Withdraw: 1>, 50, 80, 'efcab0f30bdc')
('kowalski', <OperationType.Deposit: 2>, 5, 85, 'efcab0f30bdc')
Final balance: 85
History: [(<OperationType.Deposit: 2>, 10), (<OperationType.Deposit: 2>, 20), (<OperationType.Withdraw: 1>, 50), (<OperationType.Deposit: 2>, 5)]


1. How long does it take to execute the four operations?
2. Why does it take about 4 seconds, even though the calls were submitted almost at the same time?
3. Is the final account balance correct?
4. Does the actor behave more like a function or like a process with a message queue?


## Answer

1. Its the time to execute all 4 operations on a single CPU (sequentially)
2. Actor operates on a signle thread so the execution is syncrhonous - there is no pararelization hence the 4seconds exuction time.
3. Yes the final account balance is correct because execution is sequential
4. More like a process with a message queue

Next create 4 different accounts and call single method on one of them, measure time

In [126]:
jan = BankAccount.remote("jan", initial_balance=100)
alicja = BankAccount.remote("alicja", initial_balance=100)
piotr = BankAccount.remote("piotr", initial_balance=100)
szymon = BankAccount.remote("szymon", initial_balance=100)


start = time.time()

refs = [
    jan.withdraw.remote(10),
    alicja.withdraw.remote(10),
    piotr.withdraw.remote(10),
    szymon.withdraw.remote(10),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
print("Results:")
for r in results:
    print(r)

Duration: 1.3581535816192627
Results:
('jan', <OperationType.Withdraw: 1>, 10, 90, 'efcab0f30bdc')
('alicja', <OperationType.Withdraw: 1>, 10, 90, 'efcab0f30bdc')
('piotr', <OperationType.Withdraw: 1>, 10, 90, 'efcab0f30bdc')
('szymon', <OperationType.Withdraw: 1>, 10, 90, 'efcab0f30bdc')


1. Why does it now take about 1 second instead of 4?
2. What is the unit of parallelism: the method or the actor?
3. What is the relationship between the number of actors and parallelism?


## Answer
1. Becauase each actor operates in pararell to other actors (pararelism)
2. actor
3. each actor operates on a different cpu if available

Finally lets create concurent version of the Bank account

In [127]:
account2 = BankAccount.options(max_concurrency=4).remote("nowak", initial_balance=100)

start = time.time()

refs = [
    account2.deposit.remote(10),
    account2.deposit.remote(20),
    account2.withdraw.remote(50),
    account2.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
for r in results:
    print(r)

print("Final balance:", ray.get(account2.get_balance.remote()))
print("History:", ray.get(account2.get_history.remote()))

Duration: 1.3856451511383057
('nowak', <OperationType.Deposit: 2>, 10, 85, 'efcab0f30bdc')
('nowak', <OperationType.Deposit: 2>, 20, 75, 'efcab0f30bdc')
('nowak', <OperationType.Withdraw: 1>, 50, 50, 'efcab0f30bdc')
('nowak', <OperationType.Deposit: 2>, 5, 55, 'efcab0f30bdc')
Final balance: 85
History: [(<OperationType.Withdraw: 1>, 50), (<OperationType.Deposit: 2>, 5), (<OperationType.Deposit: 2>, 20), (<OperationType.Deposit: 2>, 10)]


1. Was the execution faster?
2. Is the final state always correct?
3. Why can concurrency inside an actor be dangerous?
4. When is it better to use multiple actors, and when is it better to use a single actor with `max_concurrency`?
(https://docs.ray.io/en/latest/ray-core/api/doc/ray.actor.ActorClass.options.html)

## answer
1. Yes
2. No - the operations can be executed in pararell causing for example invalid withdrawal
3. Due to concurrent execution of actor tasks which can result in incorrect state modificaitons by actor
4. Multiple actors if the operations on a single actor (methods) in scope of actor need to follow sequential execution (like the BankAccount actor). We can use a signle Actor with max_concurrency for workloads similar to log collector in previous task in which the sequentiall execution of operations isn't a requirement

***
## Part 5 - Handling Slow Tasks

**GOAL:** The goal of this exercise is to show how to use `ray.wait` to avoid waiting for slow tasks.

See the documentation for ray.wait at https://docs.ray.io/en/latest/ray-core/api/doc/ray.wait.html.

This script starts 6 tasks, each of which takes a random amount of time to complete. We'd like to process the results in two batches (each of size 3). Change the code so that instead of waiting for a fixed set of 3 tasks to finish, we make the first batch consist of the first 3 tasks that complete. The second batch should consist of the 3 remaining tasks. Do this exercise by using `ray.wait`.

### Concepts for this Exercise - ray.wait

After launching a number of tasks, you may want to know which ones have finished executing. This can be done with `ray.wait`. The function works as follows.

```python
ready_ids, remaining_ids = ray.wait(object_ids, num_returns=1, timeout=None)
```

**Arguments:**
- `object_ids`: This is a list of object IDs.
- `num_returns`: This is maximum number of object IDs to wait for. The default value is `1`.
- `timeout`: This is the maximum amount of time in milliseconds to wait for. So `ray.wait` will block until either `num_returns` objects are ready or until `timeout` milliseconds have passed.

**Return values:**
- `ready_ids`: This is a list of object IDs that are available in the object store.
- `remaining_ids`: This is a list of the IDs that were in `object_ids` but are not in `ready_ids`, so the IDs in `ready_ids` and `remaining_ids` together make up all the IDs in `object_ids`.

Define a remote function that takes a variable amount of time to run.

In [128]:
@ray.remote
def f(i):
    np.random.seed(5 + i)
    x = np.random.uniform(0, 4)
    time.sleep(x)
    return i, time.time()

**EXERCISE:** Using `ray.wait`, change the code below so that `initial_results` consists of the outputs of the first three tasks to complete instead of the first three tasks that were submitted.

In [129]:
time.sleep(2.0)
start_time = time.time()

result_ids = [f.remote(i) for i in range(6)]
ready_ids, remaining_ids = ray.wait(result_ids, num_returns=3)
initial_results = ray.get(ready_ids)

end_time = time.time()
duration = end_time - start_time

**EXERCISE:** Change the code below so that `remaining_results` consists of the outputs of the last three tasks to complete.

In [130]:
# Wait for the remaining tasks to complete.
remaining_results = ray.get(remaining_ids)

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [131]:
assert len(initial_results) == 3
assert len(remaining_results) == 3

initial_indices = [result[0] for result in initial_results]
initial_times = [result[1] for result in initial_results]
remaining_indices = [result[0] for result in remaining_results]
remaining_times = [result[1] for result in remaining_results]

assert set(initial_indices + remaining_indices) == set(range(6))

assert duration < 1.5, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too slow.'.format(duration))

assert duration > 0.8, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too fast.'.format(duration))

# Make sure the initial results actually completed first.
assert max(initial_times) < min(remaining_times)

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.9069395065307617 seconds.


## Part 6 - Speed up Serialization

**GOAL:** The goal of this exercise is to illustrate how to speed up serialization by using `ray.put`.

### Concepts for this Exercise - ray.put

Object IDs can be created in multiple ways.
- They are returned by remote function calls.
- They are returned by actor method calls.
- They are returned by `ray.put`.

When an object is passed to `ray.put`, the object is serialized using the Apache Arrow format (see https://arrow.apache.org/ for more information about Arrow) and copied into a shared memory object store. This object will then be available to other workers on the same machine via shared memory. If it is needed by workers on another machine, it will be shipped under the hood.

**When objects are passed into a remote function, Ray puts them in the object store under the hood.** That is, if `f` is a remote function, the code

```python
x = np.zeros(1000)
f.remote(x)
```

is essentially transformed under the hood to

```python
x = np.zeros(1000)
x_id = ray.put(x)
f.remote(x_id)
```

The call to `ray.put` copies the numpy array into the shared-memory object store, from where it can be read by all of the worker processes (without additional copying). However, if you do something like

```python
for i in range(10):
    f.remote(x)
```

then 10 copies of the array will be placed into the object store. This takes up more memory in the object store than is necessary, and it also takes time to copy the array into the object store over and over. This can be made more efficient by placing the array in the object store only once as follows.

```python
x_id = ray.put(x)
for i in range(10):
    f.remote(x_id)
```

In this exercise, you will speed up the code below and reduce the memory footprint by calling `ray.put` on the neural net weights before passing them into the remote functions.

**WARNING:** This exercise requires a lot of memory to run. If this notebook is running within a Docker container, then the docker container must be started with a large shared-memory file system. This can be done by starting the docker container with the `--shm-size` flag.

In [132]:
neural_net_weights = {'variable{}'.format(i): np.random.normal(size=1000000)
                      for i in range(50)}

**EXERCISE:** Compare the time required to serialize the neural net weights and copy them into the object store using Ray versus the time required to pickle and unpickle the weights. The big win should be with the time required for *deserialization*.

Note that when you call `ray.put`, in addition to serializing the object, we are copying it into shared memory where it can be efficiently accessed by other workers on the same machine.

**NOTE:** You don't actually have to do anything here other than run the cell below and read the output.

**NOTE:** Sometimes `ray.put` can be faster than `pickle.dumps`. This is because `ray.put` leverages multiple threads when serializing large objects. Note that this is not possible with `pickle`.

In [133]:
print('Ray - serializing')
%time x_id = ray.put(neural_net_weights)
print('\nRay - deserializing')
%time x_val = ray.get(x_id)

print('\npickle - serializing')
%time serialized = pickle.dumps(neural_net_weights)
print('\npickle - deserializing')
%time deserialized = pickle.loads(serialized)

Ray - serializing
CPU times: user 0 ns, sys: 277 ms, total: 277 ms
Wall time: 119 ms

Ray - deserializing
CPU times: user 0 ns, sys: 84.8 ms, total: 84.8 ms
Wall time: 84 ms

pickle - serializing
CPU times: user 0 ns, sys: 362 ms, total: 362 ms
Wall time: 361 ms

pickle - deserializing
CPU times: user 0 ns, sys: 104 ms, total: 104 ms
Wall time: 104 ms


Define a remote function which uses the neural net weights.

In [134]:
@ray.remote
def use_weights(weights, i):
    len(weights)
    return i

**EXERCISE:** In the code below, use `ray.put` to avoid copying the neural net weights to the object store multiple times.

In [135]:
weights_ref = ray.put(neural_net_weights)

time.sleep(2.0)
start_time = time.time()

results = ray.get([
    use_weights.remote(weights_ref, i)
    for i in range(20)
])

end_time = time.time()
duration = end_time - start_time

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [136]:
assert results == list(range(20))
assert duration < 1, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.010931968688964844 seconds.


In [137]:
# Example: Parameter Server distributed application with Ray Actors
# Problem: We want to update weights and gradients, computed by workers, at a central server.
# Let's use Python class and convert that to a remote Actor class actor as a Parameter Server.
# This is a common example in machine learning where you have a central
# Parameter server updating gradients from other worker processes computing individual gradients.

print('parameter server')
@ray.remote
class ParameterSever:
    def __init__(self):
        # Initialized our gradients to zero
        self.params = np.zeros(10)

    def get_params(self):
        # Return current gradients
        return self.params

    def update_params(self, grad):
        # Update the gradients
        self.params -= grad

# Define a worker or task as a function for a remote Worker. This could be a
# machine learning function that computes gradients and sends them to
# the parameter server.

@ray.remote
def worker(ps):         # It takes an actor handle or instance as an argument
    for _ in range(5):
        time.sleep(2)
        grad = np.ones(10)
        ps.update_params.remote(grad)

# Start our Parameter Server actor. This will be scheduled as a worker process
# on a remote Ray node. You invoke its ActorClass.remote(...) to instantiate an
# Actor instance of that type.

param_server = ParameterSever.remote()
print(param_server)

# Let's get the initial values of the parameter server
print(f"Initial params: {ray.get(param_server.get_params.remote())}")

# Create Workers remote tasks computing gradients
# Let's create three separate worker tasks as our machine learning tasks
# that compute gradients. These will be scheduled as tasks on a Ray cluster.

# You can use list comprehension.
# If we need more workers to scale, we can always bump them up.
# Note: We are sending the parameter_server as an argument to the remote worker task.

print([worker.remote(param_server) for _ in range(3)])

# Now, let's iterate over a loop and query the Parameter Server as the
# workers are running independently and updating the gradients

for _i in range(20):
    print(f"Updated params: {ray.get(param_server.get_params.remote())}")
    time.sleep(1)

parameter server
Actor(ParameterSever, 0bfbbe4a252cd4a9cd0a04a001000000)
Initial params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ObjectRef(641ba3500aace403ffffffffffffffffffffffff0100000001000000), ObjectRef(fdc1a950c9d5776dffffffffffffffffffffffff0100000001000000), ObjectRef(f8f9208bc117c87affffffffffffffffffffffff0100000001000000)]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [-3. -3. -3. -3. -3. -3. -3. -3. -3. -3.]
Updated params: [-3. -3. -3. -3. -3. -3. -3. -3. -3. -3.]
Updated params: [-6. -6. -6. -6. -6. -6. -6. -6. -6. -6.]
Updated params: [-6. -6. -6. -6. -6. -6. -6. -6. -6. -6.]
Updated params: [-9. -9. -9. -9. -9. -9. -9. -9. -9. -9.]
Updated params: [-12. -12. -12. -12. -12. -12. -12. -12. -12. -12.]
Updated params: [-12. -12. -12. -12. -12. -12. -12. -12. -12. -12.]
Updated params: [-15. -15. -15. -15. -15. -15. -15. -15. -15. -15.]
Updated params: [-15. -15. -15. -

## Part 7 - Parallel merge sort (homework)
**Exercise:** Based on the previous examples, create a sequential and parallel implementation of the merge sort algorithm. You can choose any version of the parallel algorithm. Think about optimizations (when to stop spawning new workers). Compare perfromance


In [140]:
import time

def merge(a, b):
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    return out + a[i:] + b[j:]

def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    return merge(merge_sort(a[:mid]), merge_sort(a[mid:]))

@ray.remote
def sort_chunk(a):
    return merge_sort(a)

@ray.remote
def merge_task(a, b):
    return merge(a, b)

def parallel_merge_sort(a, parts=4):
    n = len(a)

    refs = [
        sort_chunk.remote(a[i*n//parts:(i+1)*n//parts])
        for i in range(parts)
    ]

    while len(refs) > 1:
        refs = [
            merge_task.remote(refs[i], refs[i + 1])
            if i + 1 < len(refs) else refs[i]
            for i in range(0, len(refs), 2)
        ]

    return ray.get(refs[0])

data = np.random.randint(0, 1_000_000, 2_000_000).tolist()

start = time.time()
seq = merge_sort(data)
seq_time = time.time() - start

start = time.time()
par = parallel_merge_sort(data, parts=4)
par_time = time.time() - start

print("Sequential time:", seq_time)
print("Parallel time:", par_time)
print("Correct:", seq == par == sorted(data))

Sequential time: 7.013063907623291
Parallel time: 2.301816701889038
Correct: True


## Part 8 - Compute Pi using Monte Carlo method (homework)
**Exercise:** Based on the previous example, create an actor-based system that can compute Pi using the Monte Carlo method. There should be one supervising actor and computing tasks/actors. As an output, we expect to have numbers approaching to 3.14


![monte-carlo](img/monte_carlo_pi_sma.png)

*Note* I should see something similar to this:

```
Current value of π is: 3.1413560033585224
Current value of π is: 3.1413457351746548
Current value of π is: 3.1413172115005907
Current value of π is: 3.1413644902634594
Current value of π is: 3.1413463306152707
Current value of π is: 3.1412993520518357
Current value of π is: 3.14122792002806
Current value of π is: 3.14125257767156
Current value of π is: 3.1412399868030354
Current value of π is: 3.14132
Current value of π is: 3.1413048336472067
Current value of π is: 3.1412723926380366
Current value of π is: 3.1413103117505994
Current value of π is: 3.141357113523027
Current value of π is: 3.1414312589618585
Current value of π is: 3.141471492704826

In [139]:
import numpy as np
import ray

@ray.remote
class PiSupervisor:
    def __init__(self):
        self.inside = 0
        self.total = 0

    def add_result(self, inside, total):
        self.inside += inside
        self.total += total

    def get_pi(self):
        return 4 * self.inside / self.total


@ray.remote
def pi_worker(supervisor, n):
    x = np.random.random(n)
    y = np.random.random(n)

    inside = np.sum(x*x + y*y <= 1)

    ray.get(supervisor.add_result.remote(int(inside), n))


supervisor = PiSupervisor.remote()

WORKERS = 4
POINTS_PER_WORKER = 100_000
ROUNDS = 10

for i in range(ROUNDS):
    refs = [
        pi_worker.remote(supervisor, POINTS_PER_WORKER)
        for _ in range(WORKERS)
    ]

    ray.get(refs)

    pi = ray.get(supervisor.get_pi.remote())
    print("Round", i + 1, "Pi estimate:", pi)

Round 1 Pi estimate: 3.14371
Round 2 Pi estimate: 3.140755
Round 3 Pi estimate: 3.14039
Round 4 Pi estimate: 3.141755
Round 5 Pi estimate: 3.141814
Round 6 Pi estimate: 3.142638333333333
Round 7 Pi estimate: 3.1429057142857144
Round 8 Pi estimate: 3.14266625
Round 9 Pi estimate: 3.14215
Round 10 Pi estimate: 3.141912


## Clean up  - Clean up the environemnt

**GOAL:** The goal of this exercise is to close the environment once you finish play with ray `ray.shutdown`.

In [99]:
import ray, gc

ray.shutdown()
gc.collect()

1753